In [5]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
using PyPlot
using PyCall
using BenchmarkTools
hp = pyimport("healpy")
include("../src/EBC.jl")

In [21]:
nside = 16
cp = gen_ConvolutionParams_pc(nside = nside);


In [4]:
lmax = nside*3-1
res = Resolution(nside)

Healpix resolution(NSIDE = 16)

In [11]:
# input beam
size = alm_idx(lmax,lmax,lmax)
blm_ = zeros(ComplexF64, 3, size)
blm = hp.blm_gauss(deg2rad(0.5), lmax = lmax, pol = true)
blm_[1,1:length(blm[1,:])] = blm[1,:]
blm_[2,1:length(blm[1,:])] = -blm[2,:]*sqrt(2)
blm_[3,1:length(blm[1,:])] = -blm[3,:]*sqrt(2);
@time blm_full = get_reorder_blm_optimized(blm_, lmax, 2);

  0.340833 seconds (692.63 k allocations: 34.412 MiB, 7.05% gc time, 99.90% compilation time)


In [10]:
alm = npzread("./inputs/alm=CMB=r0=nside$nside.npy")
@time alm_full = get_reorder_alm_optimized(alm, lmax);

  0.253864 seconds (432.26 k allocations: 21.385 MiB, 99.59% compilation time)


In [8]:
data_path = "/data/n/n339/wigner_file/"

"/data/n/n339/wigner_file/"

In [14]:
@time initial_wignerd = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time eff_wignerD = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];
#@time eff_wignerD  .= get_pc_total_effective_wignerD(cp, calc_phi[1], wd_onz[1,:], initial_wignerd, eff_wignerD);

  0.044442 seconds (41.27 k allocations: 4.405 MiB, 98.04% compilation time)
  0.051834 seconds (41.26 k allocations: 4.404 MiB, 15.41% gc time, 83.34% compilation time)


In [16]:
npix = nside2npix(nside)
utv = unique_theta_val(cp);

In [22]:
conv_binned_map = zeros(3,npix)
@time for i in 1:length(utv)
    @show i
    @show start_pix, stop_pix = unique_theta_num(i, cp)
    initial_wignerd .= initialwignerds_array(cp, utv[i], data_path, initial_wignerd);
    for pix_num in start_pix:stop_pix
        center_th, center_phi = pix2angRing(res,pix_num)
        calc_psi = rand(10)*2pi
        initial_wignerd_onz = zeros(ComplexF64, 3, 2*cp.lmax+1)
        wd_onz = effective_wignerD_onz(cp, calc_psi[:], initial_wignerd_onz)
        for i in 1:3
            eff_wignerD_2  = get_pc_total_effective_wignerD(cp, center_phi, wd_onz[i,:], initial_wignerd, eff_wignerD);
            result_d[i] = eff_convolver_optimized(alm_full, blm_full, eff_wignerD_2, 2)
        end
        conv = binned_mapmaker(calc_psi, result_d)
        conv_binned_map[:,pix_num] .= conv
    end
end

i = 1
(start_pix, stop_pix) = unique_theta_num(i, cp) = (1, 4)


LoadError: UndefVarError: `mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing Statistics in the current active module Main
Hint: a global variable of this name may be made accessible by importing StatsBase in the current active module Main

In [23]:
mean([2,3,4])

LoadError: UndefVarError: `mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing Statistics in the current active module Main
Hint: a global variable of this name may be made accessible by importing StatsBase in the current active module Main

In [6]:
xsum = rand(5)
mean(xsum)


0.3339069485475013

In [25]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using BenchmarkTools
using NPZ
using Plots
using PyPlot
using PyCall
using DataStructures

In [3]:
using BenchmarkTools
